# Focusing attention: bicycle sales in the Netherlands (2013–2023)

**Assignment:** make a visual in which attention is clearly focused on the most important data, using the NL bicycle sales dataset and the principles of *focusing attention*. The visual is made with Python (pandas + matplotlib).

**Approach in this notebook**

1. Load and understand the data
2. Decide what the *one* main message is
3. Make a quick default chart (the "before") to see what goes wrong without focus
4. Apply the principles of focusing attention step by step (the "after")
5. Explain each design choice

> Put `NL_bicycle_sales.xlsx` in the same folder as this notebook before running it.

## 1. Load the data

**What & why:** I read the Excel file into a pandas DataFrame. The column names in the file contain leading spaces (e.g. `' Electric bicycles'`), so I strip them first to avoid errors later. I also drop the empty columns.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("NL_bicycle_sales.xlsx")
df.columns = df.columns.astype(str).str.strip()     # remove leading spaces from names
df = df.dropna(axis=1, how="all")                    # drop empty columns
df = df.set_index("Year")

shares = df * 100                                    # fractions -> percentages
shares.round(1)

: 

**What it shows:** for every year from 2013 to 2023 the dataset gives the **share of total bicycle sales** per bicycle type (regular, hybrid, kids, electric, other). The values are fractions, so I converted them to percentages.

**Next step:** check that the shares add up to 100% per year, so I know this is a "part of the whole" dataset.

In [ ]:
shares.sum(axis=1).round(1)

**What it shows:** every year adds up to (almost exactly) 100%, so each row is the full Dutch bicycle market split into five types. This means a rise in one type automatically comes at the expense of the others.

**Next step:** find out which change in the data is the most important one — that is what the visual has to focus on.

## 2. What is the most important message?

**What & why:** before designing anything, I need to know *what the audience should see first*. I calculate how much each type's share changed between 2013 and 2023.

In [ ]:
change = pd.DataFrame({
    "2013 (%)": shares.loc[2013],
    "2023 (%)": shares.loc[2023],
})
change["Change (pp)"] = change["2023 (%)"] - change["2013 (%)"]
change.round(1).sort_values("Change (pp)")

**What it shows:** two types dominate the story:

* **Electric bicycles** grew from about **19% to 56%** of sales (+37 percentage points).
* **Regular bicycles** fell from **50% to 22%** (−28 pp).
* Hybrid, kids and other bicycles changed only a few points.

Between 2017 and 2018 the e-bike overtook the regular bicycle as the best-selling type, and since 2022 it is more than half of the market.

**Main message for the visual:** *"E-bikes now make up more than half of all bicycles sold in the Netherlands."* The e-bike line is the hero; the regular bicycle is the supporting character (it is what the e-bike replaced); the other three types are context.

## 3. The "before": a default chart without focus

**What & why:** to show why focusing attention matters, I first plot the data with matplotlib's default settings. My expectation: five equally strong colours, a legend and a generic title, so the reader has to work to find the message.

In [ ]:
ax = shares.plot(figsize=(9, 5), marker="o", title="Bicycle sales Netherlands")
ax.set_ylabel("Share (%)")
ax.grid(True)
plt.show()

**What it shows:** all information is technically there, but **nothing stands out**. Every line has a bright colour of equal weight, so the eye jumps between them. The reader has to look back and forth to the legend to know which line is which, the grid and markers add clutter, and the title only says *what* the data is, not *what it means*.

**Next step:** redesign the chart using the principles of focusing attention.

## 4. The "after": applying the principles of focusing attention

The principles I apply (based on *preattentive attributes* — things our brain notices in a fraction of a second, before we consciously read):

| Principle | How I apply it |
|---|---|
| **Colour used sparingly** | Only the e-bike line gets a strong colour (orange). Everything else is grey. |
| **Grey out the rest (push to background)** | Hybrid, kids and other are light grey and thin; regular bicycles darker grey because they are part of the story. |
| **Size / line weight** | The e-bike line is much thicker than the others. |
| **Enclosure / marking the key moment** | A shaded band and annotation mark the crossover (2017–2018) where e-bikes overtook regular bikes. |
| **Direct labelling instead of a legend** | Labels sit at the end of each line, so no eye movement to a legend is needed. |
| **Highlight key numbers** | Only the start and end values of the e-bike (19% → 56%) are shown, plus 22% for regular bikes. |
| **Declutter** | No grid, no top/right axis borders, no markers on grey lines, soft grey axis text. |
| **Action title** | The title states the conclusion; the subtitle explains what is measured. |
| **Text colour matches data colour** | The words "E-bikes" in the title are orange, linking text and line. |

**What & why:** the code below builds the chart in that order. My prediction: the reader sees the orange line first, reads the title, and understands the message in a few seconds.

In [ ]:
# --- Colours: one accent colour, the rest grey -------------------------
ACCENT = "#E8730C"      # orange: the hero (electric bicycles)
SUPPORT = "#6B6B6B"     # dark grey: regular bicycles (what e-bikes replaced)
CONTEXT = "#C8C8C8"     # light grey: other types (background context)
TEXT = "#4A4A4A"        # soft grey for axis text

style = {
    "Electric bicycles": dict(color=ACCENT,  lw=4.0, z=5),
    "Regular bicycles":  dict(color=SUPPORT, lw=2.2, z=4),
    "Kids bicycles":     dict(color=CONTEXT, lw=1.5, z=2),
    "Other bicycles":    dict(color=CONTEXT, lw=1.5, z=2),
    "Hybrid bicycles":   dict(color=CONTEXT, lw=1.5, z=2),
}

fig, ax = plt.subplots(figsize=(10, 6), dpi=110)
years = shares.index

# --- Draw the lines (context first, hero last so it sits on top) -------
for col, s in style.items():
    ax.plot(years, shares[col], color=s["color"], lw=s["lw"],
            zorder=s["z"], solid_capstyle="round")

# --- Mark the crossover moment ---------------------------------------
ax.axvspan(2017, 2018, color=ACCENT, alpha=0.08, zorder=0)
ax.annotate("2018: e-bikes overtake\nregular bikes",
            xy=(2017.6, 37), xytext=(2018.15, 19),
            fontsize=10, color=TEXT,
            arrowprops=dict(arrowstyle="-", color=TEXT, lw=0.8))

# --- Highlight only the key numbers ----------------------------------
e = shares["Electric bicycles"]
r = shares["Regular bicycles"]
ax.scatter([2013, 2023], [e[2013], e[2023]], color=ACCENT, s=60, zorder=6)
ax.text(2013, e[2013] - 4.5, f"{e[2013]:.0f}%", color=ACCENT,
        fontsize=12, fontweight="bold", ha="center")
ax.text(2023, e[2023] + 2.5, f"{e[2023]:.0f}%", color=ACCENT,
        fontsize=14, fontweight="bold", ha="center")
ax.scatter([2023], [r[2023]], color=SUPPORT, s=35, zorder=6)

# --- Direct labels at the end of each line (no legend) ---------------
labels = {
    "Electric bicycles": ("Electric", e[2023], ACCENT, "bold"),
    "Regular bicycles":  (f"Regular  {r[2023]:.0f}%", r[2023], SUPPORT, "bold"),
    "Kids bicycles":     ("Kids", shares.loc[2023, "Kids bicycles"] + 1.2, "#A0A0A0", "normal"),
    "Other bicycles":    ("Other", shares.loc[2023, "Other bicycles"] - 1.2, "#A0A0A0", "normal"),
    "Hybrid bicycles":   ("Hybrid", shares.loc[2023, "Hybrid bicycles"], "#A0A0A0", "normal"),
}
for col, (txt, y, c, w) in labels.items():
    ax.text(2023.35, y, txt, color=c, fontsize=11, fontweight=w, va="center")

# --- Declutter --------------------------------------------------------
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
for side in ["left", "bottom"]:
    ax.spines[side].set_color(CONTEXT)
ax.tick_params(colors=TEXT, labelsize=10, length=0)
ax.set_xticks(range(2013, 2024, 2))
ax.set_xlim(2012.7, 2024.4)
ax.set_ylim(0, 65)
ax.set_yticks([0, 20, 40, 60])
ax.set_yticklabels(["0%", "20%", "40%", "60%"])

# --- Action title + subtitle (title words coloured like the data) -----
t1 = fig.text(0.07, 0.95, "E-bikes", color=ACCENT, fontsize=17, fontweight="bold")
fig.canvas.draw()                                    # measure where "E-bikes" ends
x_end = t1.get_window_extent().transformed(fig.transFigure.inverted()).x1
fig.text(x_end + 0.006, 0.95, "now make up more than half of all bicycles sold in the Netherlands",
         color="#222222", fontsize=17, fontweight="bold")
fig.text(0.07, 0.905, "Share of total bicycle sales by type, 2013–2023",
         color=TEXT, fontsize=11)
fig.text(0.07, 0.02, "Source: NL bicycle sales dataset (course material)",
         color="#9A9A9A", fontsize=8.5)

plt.subplots_adjust(left=0.07, right=0.88, top=0.85, bottom=0.09)
fig.savefig("NL_bicycle_sales_focus.png", dpi=200, bbox_inches="tight")
plt.show()

**What it shows:** the orange e-bike line is the first thing the eye lands on. The title tells the reader the conclusion right away, the shaded band shows *when* the shift happened, and the only bold numbers are the ones that matter: **19% → 56%** for e-bikes and **22%** for regular bikes. The three small categories are still visible as light grey context, but they no longer compete for attention.

**Why it matters:** for anyone in the Dutch bicycle sector (a retailer deciding on stock, a manufacturer planning production, a municipality planning charging points or bike parking) the key takeaway is that the e-bike has become the *default* bicycle. The visual makes that clear in seconds instead of making the reader dig through five equally loud lines.

**Next step / nuance:** the e-bike share was roughly flat between 2022 (56.9%) and 2023 (56.4%). A follow-up chart could zoom in on whether the growth is levelling off.

## 5. Summary of what I did

1. **Loaded and cleaned** the data (stripped column names, dropped empty columns, converted fractions to percentages) and checked that each year adds up to 100%.
2. **Decided on one message** by calculating the change per type: the e-bike went from 19% to 56% and overtook the regular bicycle around 2018.
3. **Made a default "before" chart** to show the problem: five equally bright lines, a legend, a grid and a descriptive title — nothing stands out.
4. **Redesigned the chart** using the principles of focusing attention:
   * **colour sparingly** — one accent colour (orange) for the e-bike, everything else grey;
   * **visual hierarchy** with line thickness and shades of grey (hero → support → context);
   * **marking the key moment** with a shaded band and a short annotation;
   * **direct labels** instead of a legend;
   * **only the key numbers** labelled;
   * **decluttering** (no grid, no extra borders, soft axis text);
   * an **action title** that states the conclusion, with "E-bikes" in the same colour as the line.
5. **Saved** the final visual as `NL_bicycle_sales_focus.png`.